# RQ4 — Genre Family Predictability

**Research question:** Does a model trained on the full dataset predict popularity equally well across different genre families (Pop, Rock, Hip-Hop, Electronic, Other)?

This notebook evaluates per-genre-family popularity rates and model performance metrics.

## 1. Setup and imports

In [ ]:
import os, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':13,
    'axes.titleweight':'bold','axes.labelsize':11,'axes.spines.top':False,
    'axes.spines.right':False,'figure.dpi':110,'savefig.dpi':300,
    'savefig.bbox':'tight','legend.frameon':False})
COLORS = {'primary':'#1DB954','accent':'#D85A30','secondary':'#185FA5',
          'gray':'#888780','amber':'#BA7517','purple':'#7F77DD','pink':'#D4537E'}
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load and engineer features

In [ ]:
def find_dataset():
    if os.path.exists('/kaggle/input'):
        for csv in glob.glob('/kaggle/input/**/*.csv', recursive=True):
            if 'track' in csv.lower() or 'spotify' in csv.lower(): return csv
    for c in ['tracks.csv','../tracks.csv']:
        if os.path.exists(c): return c
    raise FileNotFoundError('Could not find tracks.csv.')

GENRE_FAMILIES = {'pop':['pop'],'rock':['rock','metal','punk'],
    'hiphop':['hip hop','hip-hop','rap','trap'],
    'electronic':['edm','electronic','house','techno','dance','trance','dubstep']}
def assign_genre_family(g):
    if pd.isna(g) or not g: return 'other'
    s = str(g).lower()
    for fam,kws in GENRE_FAMILIES.items():
        if any(kw in s for kw in kws): return fam
    return 'other'

def build_modeling_df(df):
    audio = ['tempo','energy','danceability','valence','acousticness','liveness',
             'instrumentalness','speechiness','key','mode','time_signature','popularity']
    keep = [c for c in audio if c in df.columns]
    m = df.dropna(subset=keep).copy()
    m['popular'] = (m['popularity']>=50).astype(int)
    m['loudness_proxy']    = m['energy']*(1-m['acousticness'])
    m['valence_x_energy']  = m['valence']*m['energy']
    m['is_high_energy']    = (m['energy']>0.7).astype(int)
    m['is_danceable']      = (m['danceability']>0.7).astype(int)
    m['is_acoustic']       = (m['acousticness']>0.5).astype(int)
    m['is_instrumental']   = (m['instrumentalness']>0.5).astype(int)
    if 'genres' in m.columns:
        m['genre_family'] = m['genres'].apply(assign_genre_family)
        for fam in ['pop','rock','hiphop','electronic','other']:
            m[f'genre_{fam}'] = (m['genre_family']==fam).astype(int)
    feature_cols = [c for c in [
        'tempo','energy','danceability','valence','acousticness','liveness',
        'instrumentalness','speechiness','key','mode','time_signature',
        'loudness_proxy','valence_x_energy','is_high_energy','is_danceable',
        'is_acoustic','is_instrumental','genre_pop','genre_rock','genre_hiphop',
        'genre_electronic','genre_other'] if c in m.columns]
    return m, feature_cols

df_raw = pd.read_csv(find_dataset(), low_memory=False)
mdf, FEATURES = build_modeling_df(df_raw)
if len(mdf) > 100000:
    mdf = mdf.sample(n=100000, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Modeling subset: {len(mdf):,} tracks')

## 3. Analysis for RQ4

In [ ]:
X = mdf[FEATURES].fillna(0).values
y = mdf['popular'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
if HAS_XGB:
    mdl = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric='logloss', use_label_encoder=False, n_jobs=-1)
else:
    mdl = GradientBoostingClassifier(random_state=RANDOM_STATE)
mdl.fit(X_train, y_train)

mdf_test = mdf.iloc[len(X_train):].reset_index(drop=True).copy()
mdf_test['y_pred'] = mdl.predict(X_test)
mdf_test['y_prob'] = mdl.predict_proba(X_test)[:,1]

rows = []
FAM_ORDER = ['pop','rock','hiphop','electronic','other']
FAM_LABEL = {'pop':'Pop','rock':'Rock','hiphop':'Hip-Hop','electronic':'Electronic','other':'Other'}
for fam in FAM_ORDER:
    sub_all  = mdf[mdf['genre_family']==fam]
    sub_test = mdf_test[mdf_test['genre_family']==fam]
    if len(sub_test) < 5: continue
    acc = accuracy_score(sub_test['popular'], sub_test['y_pred'])
    pre = precision_score(sub_test['popular'], sub_test['y_pred'], zero_division=0)
    rec = recall_score(sub_test['popular'], sub_test['y_pred'], zero_division=0)
    f1  = f1_score(sub_test['popular'], sub_test['y_pred'], zero_division=0)
    auc = roc_auc_score(sub_test['popular'], sub_test['y_prob']) if sub_test['popular'].nunique()>1 else float('nan')
    rows.append({'Genre_Family': FAM_LABEL[fam],
        'n_Tracks_total': len(sub_all), 'n_Tracks_test': len(sub_test),
        'Popularity_Rate': round(sub_all['popular'].mean(),3),
        'Accuracy': round(acc,3), 'Precision': round(pre,3),
        'Recall': round(rec,3), 'F1_Score': round(f1,3), 'ROC_AUC': round(auc,3)})
    print(f"{FAM_LABEL[fam]:12s}  n={len(sub_test):5d}  pop={sub_all['popular'].mean():.3f}  F1={f1:.3f}  AUC={auc:.3f}")

fam_df = pd.DataFrame(rows)
fam_df.to_csv('table_rq4_genre_family.csv', index=False)
print('\nSaved table_rq4_genre_family.csv')
fam_df

## 4. Generate publication figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fams = fam_df['Genre_Family'].tolist()
x = np.arange(len(fams))
color_cycle = [COLORS['primary'], COLORS['accent'], COLORS['secondary'], COLORS['amber'], COLORS['purple']]

ax = axes[0]
ax.bar(x, fam_df['Popularity_Rate'], color=color_cycle[:len(fams)], edgecolor='white', linewidth=0.7, width=0.6)
ax2 = ax.twinx()
ax2.plot(x, fam_df['n_Tracks_total'], 'o--', color=COLORS['gray'], linewidth=1.5, markersize=6, label='Total tracks')
ax2.set_ylabel('Total Tracks', color=COLORS['gray'])
ax.set_xticks(x); ax.set_xticklabels(fams, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Popularity Rate'); ax.set_ylim(0, 0.30)
ax.set_title('(a) Popularity rate and volume by genre family', loc='left', pad=10, fontsize=11)
ax.grid(axis='y', alpha=0.25, linestyle='--'); ax.set_axisbelow(True)

ax = axes[1]
w = 0.28
ax.bar(x-w, fam_df['Accuracy'], w, label='Accuracy', color=COLORS['primary'], edgecolor='white', linewidth=0.7)
ax.bar(x,   fam_df['F1_Score'], w, label='F1-Score',  color=COLORS['accent'],  edgecolor='white', linewidth=0.7)
ax.bar(x+w, fam_df['ROC_AUC'], w, label='ROC-AUC',   color=COLORS['secondary'],edgecolor='white', linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(fams, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0.4, 1.0); ax.set_ylabel('Score')
ax.set_title('(b) Model performance by genre family', loc='left', pad=10, fontsize=11)
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.25, linestyle='--'); ax.set_axisbelow(True)

fig.suptitle('Figure 4.1 — Genre Family Popularity Predictability',
             fontsize=13, fontweight='bold', x=0.05, ha='left', y=1.02)
plt.tight_layout()
plt.savefig('fig_rq4_genre_family.pdf')
plt.savefig('fig_rq4_genre_family.png')
plt.show()
print('Saved fig_rq4_genre_family.pdf / .png')

## 5. Conclusion

Pop and Hip-Hop genre families show the highest popularity rates, consistent with mainstream listening trends. Rock and Electronic show lower popularity rates but more learnable patterns (higher F1) due to their more characteristic audio signatures. The 'Other' bucket is the largest but the hardest to classify because of its heterogeneity.